In [41]:
import pandas as pd
import pyreadstat
import os

def build_flexible_menopause_study():
    # Update this to the exact folder name where your files sit
    path = "data"

    print(f"Current Working Directory: {os.getcwd()}")

    # Debug: Print everything inside the 'data' folder
    if os.path.exists(path):
        print(f"Files found in '{path}': {os.listdir(path)}")
    else:
        print(f"ERROR: The folder '{path}' does not exist in {os.getcwd()}")
        return None

    try:
        # Helper to join path and read XPT
        def read_nhanes(filename):
            full_path = os.path.join(path, filename)
            if not os.path.exists(full_path):
                # Try uppercase if lowercase fails, and vice versa
                alt_filename = filename.lower() if filename.isupper() else filename.upper()
                full_path = os.path.join(path, alt_filename)

            print(f"Reading: {full_path}")
            return pyreadstat.read_xport(full_path)[0]

        # 1. Load Files
        demo = read_nhanes('DEMO_C.XPT')
        mcq = read_nhanes('MCQ_C.XPT')
        rhq = read_nhanes('RHQ_C.XPT')
        l13 = read_nhanes('L13_C.XPT')

        # 2. Hard CVD Label
        cvd_cols = ['MCQ160B', 'MCQ160C', 'MCQ160D', 'MCQ160E', 'MCQ160F']
        available_cvd = [c for c in cvd_cols if c in mcq.columns]
        mcq['CVD_Label'] = mcq[available_cvd].apply(lambda x: 1 if 1.0 in x.values else 0, axis=1)

        # 3. Merge Basics
        df = pd.merge(demo[['SEQN', 'RIAGENDR', 'RIDAGEYR']], mcq[['SEQN', 'CVD_Label']], on='SEQN')
        df = df[(df['RIAGENDR'] == 2) & (df['CVD_Label'] == 1)]

        # 4. Handle LDL
        ldl_col = next((c for c in ['LBDLDL', 'LBDLDLSI', 'LBXLDL'] if c in l13.columns), None)

        if ldl_col:
            df = pd.merge(df, l13[['SEQN', 'LBXTC', ldl_col]], on='SEQN', how='inner')
            df.rename(columns={ldl_col: 'LDL'}, inplace=True)
        else:
            df = pd.merge(df, l13[['SEQN', 'LBXTC']], on='SEQN', how='inner')

        # 5. Join Reproductive Health
        df = pd.merge(df, rhq[['SEQN', 'RHQ031']], on='SEQN', how='inner')

        # 6. Apply Menopause Definition
        # Post: Age > 51 AND No period (2.0)
        # Pre: Everyone else (with CVD)
        df['Is_Post_Meno'] = ((df['RIDAGEYR'] > 51) & (df['RHQ031'] == 2.0)).astype(int)
        df['Group'] = df['Is_Post_Meno'].map({1: 'Post-menopausal', 0: 'Pre-menopausal'})

        final_df = df.dropna()

        print(f"\nAnalysis complete. Women with CVD: {len(final_df)}")
        print(final_df.groupby('Group')[['LBXTC', 'LDL' if ldl_col else 'LBXTC']].mean())

        return final_df


    except Exception as e:
        print(f"Error: {e}")
        return None


df_results = build_flexible_menopause_study()



Current Working Directory: /Users/pranav/PycharmProjects/NHANES
Files found in 'data': ['CDQ_C.xpt', 'DIQ_C.xpt', 'SMQ_C.xpt', 'L13AM_C.xpt', '.DS_Store', 'BMX_C.xpt', 'L11_C.xpt', 'L13_C.xpt', 'MCQ_C.xpt', 'DEMO_C.xpt', 'BPX_C.xpt', 'CVX_C.xpt', 'RHQ_C.xpt', 'L40_C.xpt']
Reading: data/DEMO_C.XPT
Reading: data/MCQ_C.XPT
Reading: data/RHQ_C.XPT
Reading: data/L13_C.XPT

Analysis complete. Women with CVD: 245
                      LBXTC       LBXTC
Group                                  
Post-menopausal  208.061033  208.061033
Pre-menopausal   205.656250  205.656250


In [43]:
import pandas as pd
import pyreadstat
import os

def build_refined_study():
    # Update this to your folder name
    path = "data"

    try:
        # 1. Load Files
        demo = pyreadstat.read_xport(os.path.join(path, 'DEMO_C.XPT'))[0]
        mcq = pyreadstat.read_xport(os.path.join(path, 'MCQ_C.XPT'))[0]
        rhq = pyreadstat.read_xport(os.path.join(path, 'RHQ_C.XPT'))[0]
        l13 = pyreadstat.read_xport(os.path.join(path, 'L13_C.XPT'))[0]

        # 2. Hard CVD Label
        cvd_cols = ['MCQ160B', 'MCQ160C', 'MCQ160D', 'MCQ160E', 'MCQ160F']
        available_cvd = [c for c in cvd_cols if c in mcq.columns]
        mcq['CVD_Label'] = mcq[available_cvd].apply(lambda x: 1 if 1.0 in x.values else 0, axis=1)

        # 3. Merge Basics (Females with CVD)
        df = pd.merge(demo[['SEQN', 'RIAGENDR', 'RIDAGEYR']], mcq[['SEQN', 'CVD_Label']], on='SEQN')
        df = df[(df['RIAGENDR'] == 2) & (df['CVD_Label'] == 1)]

        # 4. Handle HDL Column Naming (Dynamic Check)
        hdl_col = 'LBDHDL' if 'LBDHDL' in l13.columns else 'LBDHDLSI' if 'LBDHDLSI' in l13.columns else None

        if hdl_col:
            df = pd.merge(df, l13[['SEQN', 'LBXTC', hdl_col]], on='SEQN', how='inner')
            df.rename(columns={hdl_col: 'HDL'}, inplace=True)
            # Calculate the 'Insult' (Non-HDL)
            df['Non_HDL'] = df['LBXTC'] - df['HDL']
        else:
            print("Warning: HDL column not found. Using Total Cholesterol only.")
            df = pd.merge(df, l13[['SEQN', 'LBXTC']], on='SEQN', how='inner')

        # 5. Join Reproductive Health
        df = pd.merge(df, rhq[['SEQN', 'RHQ031']], on='SEQN', how='inner')

        # 6. Apply Menopause Groups
        # Post: Age > 51 AND No period last year (2.0)
        df['Is_Post_Meno'] = ((df['RIDAGEYR'] > 51) & (df['RHQ031'] == 2.0)).astype(int)
        df['Group'] = df['Is_Post_Meno'].map({1: 'Post-menopausal', 0: 'Pre-menopausal'})

        final_df = df.dropna()

        print(f"\nAnalysis complete. Women with CVD (N): {len(final_df)}")

        # 7. Print Results
        cols_to_show = ['LBXTC']
        if 'Non_HDL' in df.columns: cols_to_show.append('Non_HDL')

        print("\n--- RESULTS BY GROUP ---")
        print(final_df.groupby('Group')[cols_to_show].mean())

        return final_df

    except Exception as e:
        print(f"Error: {e}")
        return None

df_study = build_refined_study()


Analysis complete. Women with CVD (N): 245

--- RESULTS BY GROUP ---
                      LBXTC
Group                      
Post-menopausal  208.061033
Pre-menopausal   205.656250


In [ ]:
# Add this logic to your existing script to calculate the 'Insult'
# Total Cholesterol - HDL = Non-HDL (The 'Bad' stuff)

df_results = pd.merge(df_results, l13[['SEQN', 'LBDHDL']], on='SEQN', how='inner')
df_results['Non_HDL'] = df_results['LBXTC'] - df_results['LBDHDL']

print("\n--- Refined Mean Results (Non-HDL) ---")
print(df_results.groupby('Group')[['Non_HDL', 'LBXTC']].mean())

In [12]:
import pyreadstat
# Just load the fitness file and look at the names
cvx_peek, _ = pyreadstat.read_xport('data/CVX_C.XPT')
print("EXACT COLUMNS IN CVX_C:", cvx_peek.columns.tolist())

EXACT COLUMNS IN CVX_C: ['SEQN', 'CVDEXSTS', 'CVDEXCMT', 'CVDEXCL1', 'CVDEXCL2', 'CVDEXCL3', 'CVDEXCL4', 'CVDEXCL5', 'CVDEXCL6', 'CVQ220A', 'CVQ220B', 'CVQ220C', 'CVQ220E', 'CVQ220G', 'CVQ220H', 'CVQ220I', 'CVQ220J', 'CVQ220K', 'CVQ220L', 'CVQ220M', 'CVDEXLEN', 'CVDEXMER', 'CVDPMHR', 'CVXPARC', 'CVDVOMAX', 'CVAPROT', 'CVDPROT', 'CVATEMP', 'CVAARM', 'CVACUFF', 'CVAVEST', 'CVAWS1', 'CVAWG1', 'CVDWTIM', 'CVXWRPE', 'CVDWHR', 'CVDWSY', 'CVDWDI', 'CVDS1S', 'CVDS1G', 'CVDS1TIM', 'CVXS1RPE', 'CVDS1HR', 'CVDS1SY', 'CVDS1DI', 'CVDS2S', 'CVDS2G', 'CVDS2TIM', 'CVXS2RPE', 'CVDS2HR', 'CVDS2SY', 'CVDS2DI', 'CVDR1TIM', 'CVDR1HR', 'CVDR1SY', 'CVDR1DI', 'CVDR2TIM', 'CVDR2HR', 'CVDR2SY', 'CVDR2DI', 'CVDR3TIM', 'CVDR3HR', 'CVDR3SY', 'CVDR3DI', 'CVDESVO2', 'CVDFITLV']


In [14]:
import pandas as pd
import numpy as np

# 1. Load the dataset we just created
df = pd.read_csv('nhanes_final_with_exercise.csv')

# 2. Define the new NHANES feature groups based on your Hypothesis 1
# Exercise/Stress (Continuous physiological measurements)
exercise_features = ['Estimated_VO2_Max', 'Max_Exercise_Heart_Rate', 'Recovery_Heart_Rate']

# Resting/Baseline (Continuous lab measurements)
resting_features = ['Systolic_BP', 'Total_Cholesterol', 'Fasting_Glucose', 'C_Reactive_Protein_Inflammation']

print(f"NHANES Dataset loaded. Total participants: {len(df)}")

NHANES Dataset loaded. Total participants: 1096


In [30]:
import pandas as pd
import pyreadstat
import numpy as np

def build_crp_research_dataset():
    print("Loading data for CRP Inflammation Study...")

    # 1. Load files
    mcq = pyreadstat.read_xport('data/MCQ_C.XPT')[0]  # Heart Attack History
    cvx = pyreadstat.read_xport('data/CVX_C.XPT')[0]  # VO2, Fitness, Recovery HR
    bpx = pyreadstat.read_xport('data/BPX_C.XPT')[0]  # BP
    bmx = pyreadstat.read_xport('data/BMX_C.XPT')[0]  # BMI
    diq = pyreadstat.read_xport('data/DIQ_C.XPT')[0]  # Diabetes
    l13 = pyreadstat.read_xport('data/L13_C.XPT')[0]  # Cholesterol
    l11 = pyreadstat.read_xport('data/L11_C.XPT')[0]  # CRP (Target)

    # 2. Sequential Merge
    # Start with CRP (Target) and CVX (Core Hypothesis)
    df = pd.merge(l11[['SEQN', 'LBXCRP']],
                  cvx[['SEQN', 'CVDFITLV', 'CVDESVO2', 'CVDR1HR']], on='SEQN', how='inner')

    df = pd.merge(df, mcq[['SEQN', 'MCQ160B']], on='SEQN', how='inner')
    df = pd.merge(df, bpx[['SEQN', 'BPXSY1']], on='SEQN', how='inner')
    df = pd.merge(df, bmx[['SEQN', 'BMXBMI']], on='SEQN', how='inner')
    df = pd.merge(df, diq[['SEQN', 'DIQ010']], on='SEQN', how='inner')
    df = pd.merge(df, l13[['SEQN', 'LBXTC']],  on='SEQN', how='inner')

    # 3. Create the CRP Binary Target
    # Clinical Standard: > 3.0 mg/L = High Risk, <= 3.0 = Low/Average Risk
    df['CRP_Risk_Status'] = (df['LBXCRP'] > 1.0).astype(int)

    # 4. Rename and Clean
    rename_map = {
        'CVDFITLV': 'Fitness_Level_Category',
        'MCQ160B': 'Heart_Attack_History',
        'CVDESVO2': 'Estimated_VO2_Max',
        'CVDR1HR': 'Recovery_Heart_Rate',
        'BMXBMI': 'BMI',
        'DIQ010': 'Diabetes_Status',
        'LBXTC': 'Total_Cholesterol',
        'BPXSY1': 'Systolic_BP'
    }
    df.rename(columns=rename_map, inplace=True)

    # Convert binary mappings (1=Yes, 2=No -> 1, 0)
    for col in ['Heart_Attack_History', 'Diabetes_Status']:
        df[col] = df[col].map({1.0: 1, 2.0: 0})

    # BMI Category
    df['BMI_Category'] = pd.cut(df['BMI'], bins=[0, 18.5, 25, 30, 100], labels=[1, 2, 3, 4])

    # 5. Final Selection
    final_cols = ['CRP_Risk_Status'] + list(rename_map.values()) + ['BMI_Category']
    final_df = df[final_cols].dropna()

    final_df.to_csv('nhanes_crp_study.csv', index=False)
    print(f"Success! Final Study Size: {len(final_df)} participants.")
    print("Target Distribution (1 = High CRP):")
    print(final_df['CRP_Risk_Status'].value_counts())
    return final_df

df_final = build_crp_research_dataset()

Loading data for CRP Inflammation Study...
Success! Final Study Size: 856 participants.
Target Distribution (1 = High CRP):
CRP_Risk_Status
0    796
1     60
Name: count, dtype: int64


In [31]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency

# Load your df if not already in memory
# df = pd.read_csv('nhanes_crp_study.csv')
# df['CRP_Target'] = (df['LBXCRP'] >= 1.0).astype(int)

def calculate_cramers_v(confusion_matrix):
    chi2 = chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    phi2 = chi2 / n
    r, k = confusion_matrix.shape
    # If a table is 2x2, Cramér's V is identical to the Phi Coefficient
    return np.sqrt(phi2 / min(k - 1, r - 1))

print("--- CATEGORICAL ANALYSIS (Target: CRP >= 1.0) ---")
print(f"{'Feature':<25} | {'Group':<10} | {'Chi2':<8} | {'p-value':<10} | {'Cramér\'s V'}")
print("-" * 80)

cat_tasks = [
    ('Fitness_Level_Category', 'Exercise'),
    ('Heart_Attack_History', 'Exercise'),
    ('BMI_Category', 'Resting'),
    ('Diabetes_Status', 'Resting')
]

for feat, group in cat_tasks:
    ct = pd.crosstab(df[feat], df['CRP_Target'])
    chi2, p, _, _ = chi2_contingency(ct)
    v = calculate_cramers_v(ct)
    print(f"{feat:<25} | {group:<10} | {chi2:>8.2f} | {p:.4e} | {v:.3f}")

--- CATEGORICAL ANALYSIS (Target: CRP >= 1.0) ---
Feature                   | Group      | Chi2     | p-value    | Cramér's V
--------------------------------------------------------------------------------


KeyError: 'CRP_Target'

In [36]:
import pandas as pd
import pyreadstat
import numpy as np

def build_triglyceride_study():
    # 1. Load your standard files
    l13 = pyreadstat.read_xport('data/L13_C.XPT')[0] # Lipids
    cvx = pyreadstat.read_xport('data/CVX_C.XPT')[0] # Exercise
    bmx = pyreadstat.read_xport('data/BMX_C.XPT')[0] # BMI
    bpx = pyreadstat.read_xport('data/BPX_C.XPT')[0] # BP
    diq = pyreadstat.read_xport('data/DIQ_C.XPT')[0] # Diabetes
    mcq = pyreadstat.read_xport('data/MCQ_C.XPT')[0] # Heart History

    # 2. Merge
    # LBXTR is the code for Triglycerides
    df = pd.merge(l13[['SEQN', 'LBXTR', 'LBXTC']],
                  cvx[['SEQN', 'CVDFITLV', 'CVDESVO2', 'CVDR1HR']], on='SEQN', how='inner')

    # Add remaining features
    df = pd.merge(df, bmx[['SEQN', 'BMXBMI']], on='SEQN', how='inner')
    df = pd.merge(df, bpx[['SEQN', 'BPXSY1']], on='SEQN', how='inner')
    df = pd.merge(df, diq[['SEQN', 'DIQ010']], on='SEQN', how='inner')
    df = pd.merge(df, mcq[['SEQN', 'MCQ160B']], on='SEQN', how='inner')

    # 3. Create the Target: High Triglycerides
    # Clinical "Borderline High" starts at 150 mg/dL
    # This should give you ~150-200 people in the "1" group
    df['Triglyceride_Risk'] = (df['LBXTR'] >= 150).astype(int)

    # 4. Final Cleaning & Feature Selection
    rename_map = {
        'CVDFITLV': 'Fitness_Level_Category',
        'MCQ160B': 'Heart_Attack_History',
        'CVDESVO2': 'Estimated_VO2_Max',
        'CVDR1HR': 'Recovery_Heart_Rate',
        'BMXBMI': 'BMI',
        'DIQ010': 'Diabetes_Status',
        'LBXTC': 'Total_Cholesterol',
        'BPXSY1': 'Systolic_BP'
    }
    df.rename(columns=rename_map, inplace=True)

    # Map Binaries (1=Yes, 2=No -> 1, 0)
    for col in ['Heart_Attack_History', 'Diabetes_Status']:
        df[col] = df[col].map({1.0: 1, 2.0: 0})

    # BMI Category
    df['BMI_Category'] = pd.cut(df['BMI'], bins=[0, 18.5, 25, 30, 100], labels=[1, 2, 3, 4])

    final_df = df.dropna()
    print(f"Success! Final Study Size: {len(final_df)}")
    print("Target Distribution (1 = High Triglycerides):")
    print(final_df['Triglyceride_Risk'].value_counts())

    return final_df

df_trig = build_triglyceride_study()

KeyError: "['LBXTR'] not in index"